Assignment-9

In [ ]:
#q1
%pip install opencv-python easyocr pyttsx3
import cv2
import easyocr
import re
import difflib
import pyttsx3

# Load medicine bottle image
image = cv2.imread("medicine_bottle.jpg")

if image is None:
    raise FileNotFoundError(
        "medicine_bottle.jpg not found. Check the image path."
    )

# Initialize EasyOCR
reader = easyocr.Reader(["en"], gpu=False)

# Extract text from image
results = reader.readtext(image, detail=0)
raw_text = " ".join(results)

print("Detected text:")
print(raw_text)

# Known medicine names
KNOWN_DRUGS = [
    "paracetamol", "amoxicillin", "ibuprofen",
    "metformin", "cetirizine", "atorvastatin",
    "aspirin", "omeprazole"
]

# Match medicine name
drug_name = "Unknown medication"

for word in re.findall(r"\b[a-zA-Z]{4,}\b", raw_text.lower()):
    match = difflib.get_close_matches(
        word, KNOWN_DRUGS, n=1, cutoff=0.7
    )

    if match:
        drug_name = match[0].capitalize()
        break

# Extract dosage
dosage_match = re.search(
    r"\b\d+(?:\.\d+)?\s*(?:mg|ml|mcg|g)\b",
    raw_text,
    re.IGNORECASE
)

dosage = dosage_match.group(0) if dosage_match else "Not specified"

# Extract frequency
freq_match = re.search(
    r"\b(?:once|twice|thrice|\d+\s*times)"
    r"\s+(?:a|per)\s+day\b",
    raw_text,
    re.IGNORECASE
)

frequency = freq_match.group(0) if freq_match else "Not specified"

# Extract expiry date: MM/YY, MM/YYYY, or hyphen-separated
exp_match = re.search(
    r"\b(?:0[1-9]|1[0-2])[/\-](?:20\d{2}|\d{2})\b",
    raw_text
)

expiry = exp_match.group(0) if exp_match else "Not found"

# Display extracted information
print("\n--- Medicine Label Reader ---")
print("Medicine:", drug_name)
print("Dosage:", dosage)
print("Frequency:", frequency)
print("Expiry:", expiry)

# Speak the OCR results
advice = (
    f"Recognized medicine: {drug_name}. "
    f"Label strength: {dosage}. "
    f"Frequency text: {frequency}. "
    f"Expiry text: {expiry}."
)

engine = pyttsx3.init()
engine.say(advice)
engine.runAndWait()

In [ ]:
#q2
%pip install opencv-python deepface
import cv2
from deepface import DeepFace

# Open webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError(
        "Cannot access webcam. Check camera permissions."
    )

# Response policy
policy = {
    "happy": "You seem cheerful! Keep up the positive energy.",
    "sad": "Would you like to take a short break?",
    "angry": "Consider taking a few slow breaths.",
    "neutral": "Cognitive assistant is ready.",
    "surprise": "You seem surprised!",
    "fear": "Would a quiet moment help you relax?",
    "disgust": "Would you like to take a moment for yourself?"
}

frame_count = 0
emotion = "neutral"
message = policy[emotion]
box = None

try:
    while True:
        ret, frame = cap.read()

        if not ret:
            print("Could not read webcam frame.")
            break

        frame_count += 1

        # Analyze every 20 frames
        if frame_count % 20 == 0:
            try:
                result = DeepFace.analyze(
                    img_path=frame,
                    actions=["emotion"],
                    enforce_detection=False,
                    detector_backend="opencv",
                    silent=True
                )

                if isinstance(result, list):
                    result = result[0]

                emotion = result["dominant_emotion"]
                message = policy.get(
                    emotion,
                    "Assistant is ready."
                )

                region = result.get("region", {})
                x = int(region.get("x", 0))
                y = int(region.get("y", 0))
                w = int(region.get("w", 0))
                h = int(region.get("h", 0))

                box = (x, y, w, h) if w > 0 and h > 0 else None

            except Exception as e:
                print("Expression analysis error:", e)

        # Draw bounding box
        if box is not None:
            x, y, w, h = box
            cv2.rectangle(
                frame,
                (x, y),
                (x + w, y + h),
                (0, 255, 0),
                2
            )

        # Display expression and response
        cv2.putText(
            frame,
            f"Expression: {emotion}",
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

        # Keep message short so it fits on screen
        cv2.putText(
            frame,
            message[:55],
            (20, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 255),
            1
        )

        cv2.imshow("Emotion Cognitive Assistant", frame)

        # Press q to stop
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()